In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_all_splits, verify_data,
    CLASSES, SORTED_CLASSES, DATA_DIR,
)

print('Classes (alphabetical, index = label):')
for i, c in enumerate(SORTED_CLASSES):
    print(f'  {i} = {c}')

Classes (alphabetical, index = label):
  0 = construction_site
  1 = debris_or_rubble
  2 = fire_station
  3 = flooded_road
  4 = hospital
  5 = impoverished_settlement
  6 = multi-unit_residential
  7 = single-unit_residential


In [2]:
verify_data()

Data check: C:\Users\rdlou\class\data402\finalproject\data

processed:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
  total                       14103

processed_val:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
  total                         580

processed_test:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_all_splits(size=64)

Training set:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
Validation set:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
Test set:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        100
  single-unit_residential       200

Shapes: train (14103, 12288), val (580, 12288), test (585, 12288)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.astype(np.float32))
X_val_s   = scaler.transform(X_val.astype(np.float32))
X_test_s  = scaler.transform(X_test.astype(np.float32))

print(f'X_train_s: {X_train_s.shape}, dtype={X_train_s.dtype}')

X_train_s: (14103, 12288), dtype=float32


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best_f1, best_C, best_clf = -1, None, None

for C in [0.001, 0.01, 0.1, 1.0, 10.0]: # Doing this to see how regularized we should make the model (lower C = more simple / stronger regularization, larger C = opposite)
    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        C=C,
        n_jobs=-1,
    )
    clf.fit(X_train_s, y_train)
    val_pred = clf.predict(X_val_s)
    acc = (val_pred == y_val).mean()
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f'C={C:<7}  val acc: {acc:.3f}   macro-F1: {f1:.3f}')
    if f1 > best_f1:
        best_f1, best_C, best_clf = f1, C, clf

print(f'\nBest: C={best_C}, val macro-F1={best_f1:.3f}')

C=0.001    val acc: 0.234   macro-F1: 0.203
C=0.01     val acc: 0.228   macro-F1: 0.188
C=0.1      val acc: 0.212   macro-F1: 0.175
C=1.0      val acc: 0.184   macro-F1: 0.153
C=10.0     val acc: 0.193   macro-F1: 0.161

Best: C=0.001, val macro-F1=0.203


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = best_clf.predict(X_val_s)

print('Validation classification report:')
print(classification_report(y_val, val_pred,
                            target_names=SORTED_CLASSES, digits=3))

print('\nValidation confusion matrix:')
cm = confusion_matrix(y_val, val_pred)
print(pd.DataFrame(cm, index=SORTED_CLASSES, columns=SORTED_CLASSES))

Validation classification report:
                         precision    recall  f1-score   support

      construction_site      0.111     0.175     0.136        40
       debris_or_rubble      0.062     0.222     0.096        18
           fire_station      0.304     0.298     0.301        94
           flooded_road      0.234     0.450     0.308        40
               hospital      0.259     0.192     0.220        73
impoverished_settlement      0.045     0.125     0.067        16
 multi-unit_residential      0.216     0.192     0.203        99
single-unit_residential      0.454     0.220     0.296       200

               accuracy                          0.234       580
              macro avg      0.211     0.234     0.203       580
           weighted avg      0.302     0.234     0.249       580


Validation confusion matrix:
                         construction_site  debris_or_rubble  fire_station  \
construction_site                        7                 7             6 